# Exact versus Fourier Gram matrices: runtime comparison

For $l=1,\ldots,100$, generate a standard-normal sample $(X_1^{(l)},\ldots,X_{200}^{(l)})$. For each sample, compute the **complete** Gram matrix on the same $N=50$ geometrically spaced bandwidths in $[0.01,2]$, using both the analytic Gaussian formula and the Fourier approximation with the chosen $\Omega=450$ and $K=1536$ trapezoidal intervals ($K+1=1537$ nodes).

The bandwidth grid and sampling model follow [the repeated-sample accuracy experiment](g_rs_error_vs_k_repeated_samples.ipynb). The frequency controls are imported from [the shared defaults](../kde_tools/fourier/similarities.py). This runtime experiment uses separate seeds, `2026102600 + (l - 1)`, rather than the seeds used in the preceding accuracy experiments.

The reported quantities are our empirical summaries

$$\overline t_{\mathrm{exact}}=\frac1{100}\sum_{l=1}^{100}t_{\mathrm{exact}}^{(l)},\qquad
\overline t_{\mathrm{Fourier}}=\frac1{100}\sum_{l=1}^{100}t_{\mathrm{Fourier}}^{(l)},$$

$$E_{\max}=\max_{1\le l\le100}\max_{1\le r,s\le50}
\left|G_{h_rh_s}^{(l)}-G_{h_rh_s}^{(l),\Omega,K}\right|.$$

The maximum includes diagonal entries. The analytic matrix is the numerical reference in float64. These are measurements for the stated implementation, grid, samples, and machine.

Sources: the exact formula is [thesis Equation `eq:beta2_exact_formula02`](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex), implemented in [density.py](../kde_tools/density.py). The approximation calls [the existing Fourier helper](../kde_tools/fourier/similarities.py) without modifying either method.

Both methods receive three untimed warm-up calls on a separate sample. Measurements run sequentially, alternating which method is first (50 samples in each order). Each timed region contains one complete helper call, including its input-dependent setup and allocations; sample generation, warm-up, error calculation, result storage, and exports are excluded. We use [Python `time.perf_counter`](https://docs.python.org/3.12/library/time.html#time.perf_counter) and request one numerical-library thread with environment controls and [`threadpoolctl`](https://github.com/joblib/threadpoolctl#setting-the-maximum-size-of-thread-pools). For matching Accelerate settings, start a fresh kernel with `VECLIB_MAXIMUM_THREADS=1`; the report records the actual environment and detected thread pools.

Every run saves new measurements, both matrix stacks, samples, settings, source hashes, per-sample timings/errors, and Markdown/CSV/LaTeX summary tables under `../outputs/data/g_rs_runtime_comparison/<configuration>/<run>/`. Rerunning never substitutes cached timings. The setup also enables the required automatic PDF exporter for any subsequently displayed figures; this experiment produces tables only.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter, get_clock_info
import csv
import hashlib
import inspect
import io
import json
import os
import platform
import subprocess
import sys
import uuid

# Set before importing numerical libraries in a fresh kernel.
thread_environment_keys = (
    'VECLIB_MAXIMUM_THREADS', 'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'BLIS_NUM_THREADS',
)
for variable in thread_environment_keys:
    os.environ[variable] = '1'

code_directory = next(
    (candidate.resolve()
     for ancestor in (Path.cwd(), *Path.cwd().parents)
     for candidate in (ancestor, ancestor / 'Code' / 'Code_kde')
     if (candidate / 'kde_tools' / 'pdf_export.py').is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError('Run this notebook from inside the MASTERARBEIT project.')
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

import numpy as np
import scipy
from IPython.display import Markdown, display
from threadpoolctl import threadpool_info, threadpool_limits

from kde_tools.density import exact_gaussian_kde_l2_gram
from kde_tools.fourier import (
    DEFAULT_FREQUENCY_CUTOFF,
    DEFAULT_FREQUENCY_INTERVALS,
    FOURIER_L2_METHOD_VERSION,
    gaussian_kde_l2_gram_fourier,
)
from kde_tools.pdf_export import enable_automatic_pdf_saving

notebook_name = 'g_rs_runtime_comparison'
pdf_output_directory = enable_automatic_pdf_saving(notebook_name)

In [2]:
n = 200
realizations = 100
N = 50
h_min, h_max = 0.01, 2.0
frequency_cutoff = DEFAULT_FREQUENCY_CUTOFF
K = DEFAULT_FREQUENCY_INTERVALS
seed = 2026102600
warmup_calls_per_method = 3
absolute_tolerance = 1e-8

bandwidths = np.geomspace(h_min, h_max, N)
samples = np.stack([
    np.random.default_rng(seed + sample_index).normal(size=n)
    for sample_index in range(realizations)
])
warmup_sample = np.random.default_rng(seed + 1000000).normal(size=n)
source_paths = {
    'exact': Path(inspect.getfile(exact_gaussian_kde_l2_gram)),
    'fourier': Path(inspect.getfile(gaussian_kde_l2_gram_fourier)),
}
settings = {
    'experiment_version': 'paired_full_gram_runtime_v1',
    'sample_distribution': 'standard_normal',
    'n': n, 'realizations': realizations, 'N': N,
    'h_min': h_min, 'h_max': h_max, 'bandwidth_grid': 'geometric',
    'seed': seed, 'seed_rule': 'seed + zero_based_sample_index',
    'warmup_seed': seed + 1000000,
    'frequency_cutoff': float(frequency_cutoff), 'K': int(K),
    'frequency_points': int(K) + 1,
    'K_convention': 'number of trapezoidal intervals',
    'method_version': FOURIER_L2_METHOD_VERSION,
    'timer': 'time.perf_counter',
    'timing_scope': 'one full Gram helper call; no sampling, warm-up, validation or I/O',
    'warmup_calls_per_method': warmup_calls_per_method,
    'order': 'exact first for even zero-based indices; Fourier first for odd indices',
    'parallel_samples': False, 'requested_numerical_library_threads': 1,
    'maximum_error': 'maximum absolute difference over all samples and all ordered bandwidth pairs, including diagonal',
    'absolute_tolerance': absolute_tolerance,
    'numpy_version': np.__version__, 'scipy_version': scipy.__version__,
    'source_sha256': {
        key: hashlib.sha256(path.read_bytes()).hexdigest()
        for key, path in source_paths.items()
    },
    'sample_sha256': hashlib.sha256(samples.tobytes()).hexdigest(),
    'bandwidth_sha256': hashlib.sha256(bandwidths.tobytes()).hexdigest(),
}
configuration_id = hashlib.sha256(
    json.dumps(settings, sort_keys=True).encode()
).hexdigest()[:12]
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + uuid.uuid4().hex[:6]
data_directory = code_directory / 'outputs' / 'data' / notebook_name / configuration_id / run_id
data_directory.mkdir(parents=True, exist_ok=False)
(data_directory / 'settings.json').write_text(json.dumps(settings, indent=2) + '\n')
np.save(data_directory / 'samples.npy', samples)
np.save(data_directory / 'bandwidths.npy', bandwidths)
print(f'{realizations} samples of size {n}; {N} bandwidths; Omega={frequency_cutoff:g}; K={K}.')
print(f'Results: {data_directory}')

100 samples of size 200; 50 bandwidths; Omega=450; K=1536.
Results: /Users/iljasukhanov/MASTERARBEIT/Code/Code_kde/outputs/data/g_rs_runtime_comparison/5abef54ed2db/20261026T115648879017Z_d80c1d


In [3]:
exact_seconds = np.empty(realizations)
fourier_seconds = np.empty(realizations)
exact_grams = np.empty((realizations, N, N))
fourier_grams = np.empty_like(exact_grams)
per_sample_max_errors = np.empty(realizations)
worst_indices = np.empty((realizations, 2), dtype=int)

# No other sample calculations run concurrently with these measurements.
with threadpool_limits(limits=1):
    for _ in range(warmup_calls_per_method):
        exact_gaussian_kde_l2_gram(warmup_sample, bandwidths)
        gaussian_kde_l2_gram_fourier(
            warmup_sample, bandwidths,
            frequency_cutoff=frequency_cutoff, frequency_points=int(K) + 1,
        )
    active_threadpools = threadpool_info()
    experiment_start = perf_counter()
    for sample_index, current_sample in enumerate(samples):
        method_order = ('exact', 'fourier') if sample_index % 2 == 0 else ('fourier', 'exact')
        for method in method_order:
            if method == 'exact':
                start = perf_counter()
                current_exact = exact_gaussian_kde_l2_gram(current_sample, bandwidths)
                duration = perf_counter() - start
                exact_seconds[sample_index] = duration
                exact_grams[sample_index] = current_exact
            else:
                start = perf_counter()
                current_fourier = gaussian_kde_l2_gram_fourier(
                    current_sample, bandwidths,
                    frequency_cutoff=frequency_cutoff, frequency_points=int(K) + 1,
                )
                duration = perf_counter() - start
                fourier_seconds[sample_index] = duration
                fourier_grams[sample_index] = current_fourier
        difference = np.abs(current_fourier - current_exact)
        r, s = np.unravel_index(np.argmax(difference), difference.shape)
        worst_indices[sample_index] = (r, s)
        per_sample_max_errors[sample_index] = difference[r, s]
        if (sample_index + 1) % 10 == 0:
            print(f'Completed {sample_index + 1}/{realizations} samples '
                  f'({perf_counter() - experiment_start:.1f} s).', flush=True)
    experiment_seconds = perf_counter() - experiment_start

np.savez_compressed(
    data_directory / 'per_sample_results.npz',
    exact_seconds=exact_seconds, fourier_seconds=fourier_seconds,
    maximum_absolute_errors=per_sample_max_errors, worst_indices=worst_indices,
    exact_grams=exact_grams, fourier_grams=fourier_grams,
)
with (data_directory / 'per_sample_results.csv').open('w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow([
        'sample_l_one_based', 'seed', 'first_method', 'exact_seconds',
        'fourier_seconds', 'maximum_absolute_error', 'worst_r_zero_based',
        'worst_s_zero_based', 'h_r', 'h_s',
    ])
    for sample_index in range(realizations):
        r, s = worst_indices[sample_index]
        writer.writerow([
            sample_index + 1, seed + sample_index,
            'exact' if sample_index % 2 == 0 else 'fourier',
            exact_seconds[sample_index], fourier_seconds[sample_index],
            per_sample_max_errors[sample_index], int(r), int(s),
            bandwidths[r], bandwidths[s],
        ])

Completed 10/100 samples (3.1 s).
Completed 20/100 samples (6.2 s).
Completed 30/100 samples (9.3 s).
Completed 40/100 samples (12.3 s).
Completed 50/100 samples (15.4 s).
Completed 60/100 samples (18.6 s).
Completed 70/100 samples (21.6 s).
Completed 80/100 samples (24.6 s).
Completed 90/100 samples (27.7 s).
Completed 100/100 samples (30.8 s).


In [4]:
# Verify the saved measurements and all entries before writing the table.
assert exact_grams.shape == fourier_grams.shape == (realizations, N, N)
assert np.isfinite(exact_grams).all() and np.isfinite(fourier_grams).all()
assert np.isfinite(exact_seconds).all() and np.all(exact_seconds > 0)
assert np.isfinite(fourier_seconds).all() and np.all(fourier_seconds > 0)
assert np.array_equal(exact_grams, exact_grams.transpose(0, 2, 1))
assert np.allclose(fourier_grams, fourier_grams.transpose(0, 2, 1), rtol=0, atol=1e-14)
all_errors = np.abs(fourier_grams - exact_grams)
assert np.array_equal(per_sample_max_errors, all_errors.max(axis=(1, 2)))

worst_l, worst_r, worst_s = np.unravel_index(np.argmax(all_errors), all_errors.shape)
maximum_error = float(all_errors[worst_l, worst_r, worst_s])
mean_exact_seconds = float(exact_seconds.mean())
mean_fourier_seconds = float(fourier_seconds.mean())
speedup = mean_exact_seconds / mean_fourier_seconds

def timing_statistics(values):
    return {
        'mean_seconds': float(values.mean()),
        'standard_deviation_seconds': float(values.std(ddof=1)),
        'median_seconds': float(np.median(values)),
        'minimum_seconds': float(values.min()),
        'maximum_seconds': float(values.max()),
        'total_seconds': float(values.sum()),
    }

cpu_model = platform.processor()
if sys.platform == 'darwin':
    cpu_query = subprocess.run(
        ['sysctl', '-n', 'machdep.cpu.brand_string'],
        capture_output=True, text=True, check=False,
    )
    if cpu_query.returncode == 0:
        cpu_model = cpu_query.stdout.strip()

report = {
    **settings, 'configuration_id': configuration_id, 'run_id': run_id,
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'platform': platform.platform(), 'machine': platform.machine(),
    'cpu_model': cpu_model, 'logical_cpu_count': os.cpu_count(),
    'python_version': sys.version, 'python_executable': sys.executable,
    'clock_info': vars(get_clock_info('perf_counter')),
    'thread_environment': {key: os.environ.get(key) for key in thread_environment_keys},
    'detected_threadpools_during_timing': active_threadpools,
    'exact': timing_statistics(exact_seconds),
    'fourier': timing_statistics(fourier_seconds),
    'speedup_ratio_of_mean_times': speedup,
    'maximum_absolute_error': maximum_error,
    'samples_exceeding_tolerance': int(np.count_nonzero(per_sample_max_errors > absolute_tolerance)),
    'worst_case': {
        'sample_l_one_based': int(worst_l) + 1, 'seed': seed + int(worst_l),
        'r_zero_based': int(worst_r), 's_zero_based': int(worst_s),
        'h_r': float(bandwidths[worst_r]), 'h_s': float(bandwidths[worst_s]),
        'exact_value': float(exact_grams[worst_l, worst_r, worst_s]),
        'fourier_value': float(fourier_grams[worst_l, worst_r, worst_s]),
    },
    'ordered_pairs_per_sample_including_diagonal': N * N,
    'unique_symmetric_pairs_per_sample_including_diagonal': N * (N + 1) // 2,
    'checked_entries_total': realizations * N * N,
    'experiment_elapsed_seconds_excluding_warmup_and_exports': experiment_seconds,
    'timing_order_counts': {'exact_first': (realizations + 1) // 2, 'fourier_first': realizations // 2},
    'validation': {
        'all_entries_finite': True, 'all_times_finite_and_positive': True,
        'exact_matrices_symmetric': True, 'fourier_matrices_symmetric_to_1e_minus_14': True,
        'stored_per_sample_maxima_match_full_matrices': True,
    },
}
(data_directory / 'report.json').write_text(json.dumps(report, indent=2) + '\n')
np_config_buffer = io.StringIO()
from contextlib import redirect_stdout
with redirect_stdout(np_config_buffer):
    np.show_config()
(data_directory / 'numpy_configuration.txt').write_text(np_config_buffer.getvalue())

with (data_directory / 'summary.csv').open('w', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['method', 'mean_seconds', 'mean_milliseconds', 'standard_deviation_seconds', 'maximum_absolute_error_vs_exact'])
    writer.writerow(['Exact Gaussian formula', mean_exact_seconds, 1000 * mean_exact_seconds, report['exact']['standard_deviation_seconds'], ''])
    writer.writerow(['Fourier approximation', mean_fourier_seconds, 1000 * mean_fourier_seconds, report['fourier']['standard_deviation_seconds'], maximum_error])

summary_markdown = (
    '| Method | Mean time per complete matrix (ms) | Maximum absolute error |\n'
    '| --- | ---: | ---: |\n'
    f'| Exact Gaussian formula | {1000 * mean_exact_seconds:.3f} | Reference |\n'
    f'| Fourier approximation | {1000 * mean_fourier_seconds:.3f} | {maximum_error:.3e} |\n'
)
summary_note = (
    f'\n100 standard-normal samples, n=200; N=50 geometric bandwidths in [0.01, 2]; '
    f'Omega={frequency_cutoff:g}, K={K} intervals. '
    f'The Fourier implementation is {speedup:.2f} times faster by the ratio of mean times. '
    f'The error is the maximum over all {realizations * N * N:,} matrix entries, including diagonals. '
    f'Measured on {cpu_model} ({platform.system()} {platform.release()}); timings are machine dependent.\n'
)
(data_directory / 'summary.md').write_text(summary_markdown + summary_note)

mantissa, exponent = f'{maximum_error:.3e}'.split('e')
latex_error = rf'${mantissa}\times10^{{{int(exponent)}}}$'
latex_table = '\n'.join([
    r'\begin{table}[htbp]', r'\centering', r'\begin{tabular}{lrr}', r'\hline',
    r'Method & Mean time (ms) & Maximum absolute error \\', r'\hline',
    f'Exact Gaussian formula & {1000 * mean_exact_seconds:.3f} & --- ' + r'\\',
    f'Fourier approximation & {1000 * mean_fourier_seconds:.3f} & {latex_error} ' + r'\\',
    r'\hline', r'\end{tabular}',
    rf'\caption{{Runtime comparison over $M={realizations}$ standard-normal samples of size $n={n}$, '
    rf'using $N={N}$ geometrically spaced bandwidths in $[0.01,2]$, '
    rf'$\Omega={frequency_cutoff:g}$ and $K={K}$ trapezoidal intervals. '
    r'Times are arithmetic means for complete Gram-matrix calculations. '
    r'The maximum absolute error is taken over all samples and bandwidth pairs, including diagonal entries, '
    r'with the float64 analytic calculation as reference.}',
    r'\label{tab:exact_fourier_gram_runtime}', r'\end{table}', '',
])
(data_directory / 'runtime_comparison_table.tex').write_text(latex_table)
(code_directory / 'outputs' / 'data' / notebook_name / 'latest_run.json').write_text(
    json.dumps({'relative_directory': str(data_directory.relative_to(code_directory)),
                'configuration_id': configuration_id, 'run_id': run_id}, indent=2) + '\n'
)
display(Markdown(summary_markdown + summary_note))
print(f'Worst error: {maximum_error:.12e}; sample l={worst_l + 1}, h={bandwidths[worst_r]:g}, s={bandwidths[worst_s]:g}.')
print(f'Samples above {absolute_tolerance:g}: {report["samples_exceeding_tolerance"]}/{realizations}.')
print(f'LaTeX table: {data_directory / "runtime_comparison_table.tex"}')

| Method | Mean time per complete matrix (ms) | Maximum absolute error |
| --- | ---: | ---: |
| Exact Gaussian formula | 297.500 | Reference |
| Fourier approximation | 10.111 | 9.182e-11 |

100 standard-normal samples, n=200; N=50 geometric bandwidths in [0.01, 2]; Omega=450, K=1536 intervals. The Fourier implementation is 29.42 times faster by the ratio of mean times. The error is the maximum over all 250,000 matrix entries, including diagonals. Measured on arm (Darwin 25.5.0); timings are machine dependent.


Worst error: 9.181812254955e-11; sample l=44, h=2, s=2.
Samples above 1e-08: 0/100.
LaTeX table: /Users/iljasukhanov/MASTERARBEIT/Code/Code_kde/outputs/data/g_rs_runtime_comparison/5abef54ed2db/20261026T115648879017Z_d80c1d/runtime_comparison_table.tex
